<!-- nav -->
[← 8.05 · TypeScript Patterns](08_05_TypeScript_Patterns.ipynb) · [Home](../../README.md) · [8.07 · C Types →](08_07_C_Types.ipynb)

# 8.06 · TypeScript RegExp

TypeScript writes a regular expression as a literal, between slashes, its flags after:

```text
const repeated = /(?<word>[a-z]+)-\k<word>/giv;
```

ECMA-262 defines what is between the slashes, its pattern grammar (22.2.1, "Patterns"), and the flags. `TypeScript
RegExp` is that grammar's dialect, its kinds regexpp's node types, ECMAScript's reference parser's, lowercased:
`regexpliteral`, `pattern`, `alternative`, `capturinggroup`, `quantifier`, `characterclass` and the rest, with two
helpers, a literal's `Flags` and a group's `ModifierFlags`
([CAP-281: Python's, ECMAScript's and C++'s regular expressions as data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-281.md)).
A regular expression literal is TypeScript5's `constant` (tutorial 3, 03.7), which holds its spelling, `pattern` and
`flags`, beside the regular expression they encode, its `regexp`, a term of this dialect.

This page builds regular expressions from a single character up, in the builders' fluent calls, prints each as
ECMAScript writes it, puts one in TypeScript5's literal, validates them, and reads the editions of ECMA-262 that have
its kinds. Like the rest of Part I, it starts from `dialects()`; `regexp` holds a spec in the dialect's holder,
`S.TypeScript.RegExp.Expression`, `char` writes a character and `seq` an alternative of elements. No reader here reads
a regular expression's text, in either language: reading TypeScript is mbse-programs' (tutorial 3).

In [1]:
from mbse.Expressions.Framework import Standards, Symbolics, Terms
from mbse.Expressions.TypeScript import RegExp as TypeScriptRegExp, TypeScript5
from toolkit import dialects
from mbse.Expressions.Framework.Simplifiers import NotEvaluated

store = dialects()
S = store.Expressions
REGEXP = TypeScriptRegExp.Expressions.DIALECT
ToText = TypeScriptRegExp.Text.ToText
char = lambda c: lambda e: e.character(lambda h: h.value(ord(c)))  # the character c, its value a code point
seq = lambda *elements: lambda e: e.alternative(lambda a: a.elements(list(elements)))  # an alternative of elements


def regexp(spec):
    """The regular expression `spec` writes, held by its dialect's holder, written in a transaction of its own."""
    with store.transaction():
        return S.TypeScript.RegExp.Expression(lambda e: e.root(spec))


def show(held):
    """A regular expression's text, as ECMAScript writes it, and its kind."""
    print(f"{ToText(held):32}", Terms.kind_of(Terms.term(held))[0])

<a id="T08.06.1"></a>
## 08.06.1 · A character, a pattern, a literal

The smallest regular expression is one character, `a`, its `value` a code point. Characters in a row are an
`alternative`; alternatives joined by `|` a `pattern`; and a pattern between slashes, with its flags, a `regexpliteral`,
the flags a `Flags` helper of ECMAScript's eight, `d`, `g`, `i`, `m`, `s`, `u`, `v` and `y`, each a boolean named as
regexpp names it (`isGlobal` for `g`, `ignoreCase` for `i`):

In [2]:
pattern = lambda e: e.pattern(lambda p: p.alternatives([seq(char("a"), char("b")), seq(char("c"))]))  # ab|c
for spec in (char("a"), seq(char("a"), char("b")), pattern,
             lambda r: r.regexpliteral(lambda l: l.pattern(pattern).flags(lambda f: f.isGlobal(True).ignoreCase(True)))):
    show(regexp(spec))

a                                character
ab                               alternative
ab|c                             pattern
/ab|c/gi                         regexpliteral


<a id="T08.06.2"></a>
## 08.06.2 · Classes, sets and quantifiers

A **character class**, `[a-z_]`, matches one of its elements, characters and ranges, `[^...]` none of them. A
**character set** is a class ECMAScript names: `.`, `\d`, `\s`, `\w` and their negations, each a `characterset` of its
`setKind`, and a Unicode property, `\p{Script=Greek}`, its `key` and `value`. A **quantifier** repeats its `element`
between its `min` and its `max`; the printer writes its shortest spelling, `*`, `+`, `?` or `{2,5}`, then `?` where it is
lazy:

In [3]:
lowercase = lambda e: e.characterclassrange(lambda r: r.min(char("a")).max(char("z")))
specs = [
    lambda r: r.characterclass(lambda c: c.elements([lowercase, char("_")])),
    lambda r: r.characterclass(lambda c: c.negate(True).elements([lowercase])),
    lambda r: r.characterset(lambda s: s.setKind("any")),
    lambda r: r.characterset(lambda s: s.setKind("digit")),
    lambda r: r.characterset(lambda s: s.setKind("word").negate(True)),
    lambda r: r.characterset(lambda s: s.setKind("property").key("Script").value("Greek")),
    lambda r: r.quantifier(lambda q: q.min(0).element(char("a"))),
    lambda r: r.quantifier(lambda q: q.min(1).greedy(False).element(lambda e: e.characterset(lambda s: s.setKind("digit")))),
    lambda r: r.quantifier(lambda q: q.min(2).max(5).element(char("a"))),
]
for spec in specs:
    show(regexp(spec))

[a-z_]                           characterclass
[^a-z]                           characterclass
.                                characterset
\d                               characterset
\W                               characterset
\p{Script=Greek}                 characterset
a*                               quantifier
\d+?                             quantifier
a{2,5}                           quantifier


<a id="T08.06.3"></a>
## 08.06.3 · Groups, assertions, references and modifiers

A **group**, `(?:a|b)`, gathers alternatives; a **capturing group**, `(a)` or `(?<word>a)`, numbered as it opens and
perhaps named; a **backreference**, `\1` or `\k<word>`, matches what one matched, its `ref` a number or a name. An
**assertion** matches a place: `^`, `$`, `\b`, and the lookarounds, `(?=a)` and `(?<!a)`, which hold alternatives. Two
editions added more: the `v` flag's class expressions (ES2024), `[\w--a]` a word character but `a`, an
`expressioncharacterclass` of a `classsubtraction`; and **modifiers** (ES2025), `(?i:a)`, a group's `modifiers`, which
add or remove flags within it. The opening example, whole:

In [4]:
word = lambda e: e.capturinggroup(lambda g: g.name("word").alternatives([seq(
    lambda e: e.quantifier(lambda q: q.min(1).element(lambda c: c.characterclass(lambda k: k.elements([lowercase])))))]))
repeated = regexp(lambda r: r.regexpliteral(lambda l: l
    .pattern(lambda e: e.pattern(lambda p: p.alternatives([seq(word, char("-"), lambda e: e.backreference(lambda b: b.ref("word")))])))
    .flags(lambda f: f.isGlobal(True).ignoreCase(True).unicodeSets(True))))
show(repeated)
specs = [
    lambda r: r.group(lambda g: g.alternatives([seq(char("a")), seq(char("b"))])),
    seq(lambda e: e.capturinggroup(lambda g: g.alternatives([seq(char("a"))])), lambda e: e.backreference(lambda b: b.ref(1))),
    seq(lambda e: e.assertion(lambda a: a.assertionKind("start")), char("a"),
        lambda e: e.assertion(lambda a: a.assertionKind("word")),
        lambda e: e.assertion(lambda a: a.assertionKind("lookahead").alternatives([seq(char("b"))])),
        lambda e: e.assertion(lambda a: a.assertionKind("lookbehind").negate(True).alternatives([seq(char("c"))])),
        lambda e: e.assertion(lambda a: a.assertionKind("end"))),
    lambda r: r.expressioncharacterclass(lambda x: x.expression(lambda e: e.classsubtraction(lambda s: s
        .left(lambda w: w.characterset(lambda t: t.setKind("word"))).right(char("a"))))),
    lambda r: r.group(lambda g: g.modifiers(lambda m: m.modifiers(lambda s: s.add(lambda a: a.ignoreCase(True))))
        .alternatives([seq(char("a"))])),
]
for spec in specs:
    show(regexp(spec))

/(?<word>[a-z]+)-\k<word>/giv    regexpliteral
(?:a|b)                          group
(a)\1                            alternative
^a\b(?=b)(?<!c)$                 alternative
[\w--a]                          expressioncharacterclass
(?i:a)                           group


<a id="T08.06.4"></a>
## 08.06.4 · A spelling kept beside what it encodes

A reader keeps how a regular expression was written where the tree does not give it: a character's `raw`, `\x41` for
65, a property's, `\p{L}` for the general category `L`, a quantifier's, a backreference's and the flags'. The printer
writes a `raw` back as written, and, where there is none, the default spelling: a character escaped where it is
syntax, `\.`, a property by its key and value
([PRI-023: domain representations](https://github.com/pitaman71/mbse-basis/blob/main/docs/PRINCIPLES.md#PRI-023)):

In [5]:
letter = lambda s: s.setKind("property").key("General_Category").value("L")
specs = [
    lambda r: r.character(lambda c: c.raw("\\x41").value(65)), lambda r: r.character(lambda c: c.value(65)),
    char("."),
    lambda r: r.characterset(lambda s: letter(s.raw("\\p{L}"))), lambda r: r.characterset(letter),
    lambda r: r.quantifier(lambda q: q.raw("a{0,}").min(0).element(char("a"))),
]
for spec in specs:
    held = regexp(spec)
    attributes = Terms.form(Terms.term(held)).attributes
    print(f"{ToText(held):32}", f"{Terms.kind_of(Terms.term(held))[0]:13}", ", ".join(f"{k}={v}" for k, v in attributes.items()))

\x41                             character     raw=\x41, value=65
A                                character     value=65
\.                               character     value=46
\p{L}                            characterset  raw=\p{L}, setKind=property, key=General_Category, value=L
\p{General_Category=L}           characterset  setKind=property, key=General_Category, value=L
a{0,}                            quantifier    raw=a{0,}, min=0


<a id="T08.06.5"></a>
## 08.06.5 · In TypeScript5's literal

TypeScript5's `constant` is a regular expression literal: its `pattern` and `flags` the spelling a reader read, and its
`regexp` the regular expression they encode, a subterm of `TypeScript RegExp`' `Term`, the one property of the family
that holds one. A constant written from code may hold its `regexp` alone, and the printer writes it from that.
TypeScript5 validates the constant with its regular expression, infers its domain, `RegExp`, and, as it folds its literals alone, leaves it in the residual, named. A string given to `new RegExp("a+")` is a string, not this dialect's term: no reader guesses
it is one ([PRI-025: never guess the schema](https://github.com/pitaman71/mbse-basis/blob/main/docs/PRINCIPLES.md#PRI-025)):

In [6]:
literal = lambda r: r.regexpliteral(lambda l: l.pattern(lambda e: e.pattern(lambda p: p.alternatives([seq(
    lambda e: e.quantifier(lambda q: q.min(1).element(char("a"))))]))).flags(lambda f: f.raw("g").isGlobal(True)))  # /a+/g
with store.transaction():
    spelled = S.TypeScript.TypeScript5.Expression(lambda e: e.root(lambda u: u.constant(lambda c: c.pattern("a+").flags("g").regexp(literal))))
    written = S.TypeScript.TypeScript5.Expression(lambda e: e.root(lambda u: u.constant(lambda c: c.regexp(literal))))
    called = S.TypeScript.TypeScript5.Expression(lambda e: e.root(lambda u: u.newexpression(lambda n: n
        .callee(lambda c: c.identifier(lambda i: i.name("RegExp")))
        .arguments([lambda a: a.literal(lambda l: l.value(lambda v: v.as_scalar("a+")).quote('"'))]))))
dialect = TypeScript5.Expressions.DIALECT
for held in (spelled, written):
    form = Terms.form(Terms.term(held))
    print(f"{TypeScript5.Text.ToText(held):8}", form.kind, ", ".join(f"{k}={v}" for k, v in form.attributes.items()) or "-", "| regexp:", Terms.kind_of(form.arguments[0])[0],
          "| problems:", "; ".join(dialect.validate(held)) or "none", "| domain:", dialect.infer(held).name())
try:
    TypeScript5.Simplifiers.OfAny(spelled).evaluate()
    raise AssertionError("evaluated")
except NotEvaluated as error:
    print(error)
print(TypeScript5.Text.ToText(called), "| its terms' dialects:", ", ".join(sorted({Terms.dialect_of(t) for t in Terms.walk(Terms.term(called))})))

/a+/g    constant pattern=a+, flags=g | regexp: regexpliteral | problems: none | domain: RegExp
/a+/g    constant - | regexp: regexpliteral | problems: none | domain: RegExp
TypeScript5 constant is not evaluated
new RegExp("a+") | its terms' dialects: Expressions.TypeScript.TypeScript5.Term


<a id="T08.06.6"></a>
## 08.06.6 · Validated, binding nothing, not simplified

Validation reports ECMA-262's early errors that a term's data can show, in its own words: a quantifier's bounds out of
order, flags given twice, both `u` and `v`, a modifier both added and removed, a range out of order. A named group's
name is a label its backreferences resolve, not a name the expression binds: a regular expression needs nothing from a
scope. And it is matched by the engine, not simplified: TypeScript5's rules fold none of its kinds: each stays in the residual, named by its dialect:

In [7]:
refused = [
    lambda r: r.quantifier(lambda q: q.min(2).max(1).element(char("a"))),
    lambda r: r.regexpliteral(lambda l: l.pattern(lambda e: e.pattern(lambda p: p.alternatives([seq(char("a"))])))
        .flags(lambda f: f.raw("gg").isGlobal(True))),
    lambda r: r.regexpliteral(lambda l: l.pattern(lambda e: e.pattern(lambda p: p.alternatives([seq(char("a"))])))
        .flags(lambda f: f.unicode(True).unicodeSets(True))),
    lambda r: r.characterclassrange(lambda c: c.min(char("z")).max(char("a"))),
]
for spec in refused:
    held = regexp(spec)
    print(f"{ToText(held):10}", "; ".join(REGEXP.validate(held)))
print(ToText(repeated), "| free:", ", ".join(Symbolics.free(repeated, REGEXP)) or "none", "| problems:", "; ".join(REGEXP.validate(repeated)) or "none")
try:
    TypeScript5.Simplifiers.OfAny(repeated).evaluate()
    raise AssertionError("evaluated")
except NotEvaluated as error:
    print(error)

a{2,1}     a quantifier's max 1 is less than its min 2
/a/gg      a regular expression's flags cannot be 'gg'
/a/uv      a regular expression's flags cannot have both u and v
z-a        a characterclassrange's min 122 is greater than its max 97
/(?<word>[a-z]+)-\k<word>/giv | free: none | problems: none
TypeScript RegExp regexpliteral is not evaluated


<a id="T08.06.7"></a>
## 08.06.7 · Its editions as data

`TypeScript RegExp` covers ECMA-262's editions from the 5.1 Edition to the 17th, each found by Ecma International's
designation or a common name, `ES2015` or `ECMAScript 2024`
([CAP-268: standards as tagged data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-268.md)),
and records which of its kinds each has: the `v` flag's five from the 15th Edition (ES2024), `modifiers` from the 16th
(ES2025); the 17th's pattern grammar is the 16th's:

In [8]:
dialect = store.Instances.Expressions.TypeScript.RegExp
editions = Standards.standards(dialect)
print(len(editions), "editions:", list(editions)[0], "to", list(editions)[-1])
for tag in ("ES5.1", "ES2023", "ES2024", "ES2025", "ES2026"):
    edition = Standards.find(dialect, tag)
    print(f"{tag:7}", f"{Standards.tags(edition)[0]:21}", len(Standards.kinds(edition)), "kinds")
modifiers = store.registered("Expressions.TypeScript.RegExp.Modifiers")
print("Modifiers since", Standards.tags(next(iter(Standards.of(dialect, modifiers).values())))[-1])

13 editions: ECMA-262 5.1 Edition to ECMA-262 17th Edition
ES5.1   ECMA-262 5.1 Edition  12 kinds
ES2023  ECMA-262 14th Edition 12 kinds
ES2024  ECMA-262 15th Edition 17 kinds
ES2025  ECMA-262 16th Edition 18 kinds
ES2026  ECMA-262 17th Edition 18 kinds
Modifiers since ES2025


## What you learned

- **A regular expression is a term of its own dialect**, `TypeScript RegExp`, regexpp's node types as kinds, from a
  character up to a literal with its flags.
- **TypeScript5's literal holds one** as its `regexp`, beside its spelling, the one place of the family that does; a
  string given to `new RegExp` stays a string.
- **A spelling is kept beside what it encodes**, and the printer writes it back, or a default spelling.
- **Validation reports ECMA-262's early errors**; a named group's name is a label, so it binds nothing; simplification leaves each kind in the residual, named.
- **Its editions are data**: ECMA-262's 5.1 Edition to the 17th, the `v` flag's classes and modifiers where they began.

That ends the Python and TypeScript families' sublanguages. Next, Part II puts expressions to work as constraints,
starting with case study 9.

<!-- nav -->
[← 8.05 · TypeScript Patterns](08_05_TypeScript_Patterns.ipynb) · [Home](../../README.md) · [8.07 · C Types →](08_07_C_Types.ipynb)